Requirements

1. python 3.11

2. NI-MAX

3. pip install pyvisa

4. pip install git+https://github.com/OE-FET/keithley2600

In [ ]:
# %pip install git+https://github.com/OE-FET/keithley2600

  Cloning https://github.com/OE-FET/keithley2600 to c:\users\20245580\appdata\local\temp\pip-req-build-rvsdwdvq
  Resolved https://github.com/OE-FET/keithley2600 to commit f0434ef683e053d9ebbfa18415ac3ce6dee4e78b
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ---------------------------------------- 13.1/13.1 MB 74.6 MB/s eta 0:00:00
  Created wheel for keithley2600: filename=keithley2600-2.1.0-py3-none-any.whl size=26518 sha256=26b026199506d315b283cb63a7cfc0151dd079788cc8baf329004507fa549016
  Stored in directory: C:\Users\20245580\AppData\Local\Temp\pip-ephem-wheel-cache-cwjae7g3\wheels\b0\47\b8\ef552df3892f3c1af6872b21992963535d203cac45d003eaa8
Successfully built keithley2600

   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ------------

  Running command git clone --filter=blob:none --quiet https://github.com/OE-FET/keithley2600 'C:\Users\20245580\AppData\Local\Temp\pip-req-build-rvsdwdvq'
  DEPRECATION: Building 'keithley2600' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'keithley2600'. Discussion can be found at https://github.com/pypa/pip/issues/6334


# keithley2600

Fail

In [80]:
import pyvisa
from  keithley2600 import Keithley2600
import time
import logging
import csv
import os
from datetime import datetime

    # # ======
    # # Logger
    # # ======
# init logger
format = "%(asctime)s: %(message)s"
log_file_path = 'example.log'
logging.basicConfig(format=format, level=logging.INFO,  
                        datefmt="%H:%M:%S", filename= log_file_path, filemode= 'w')

    # # ======
    # # Keithley
    # # ======
# init logger
keithley_instrument = Keithley2600('USB0::0x05E6::0x2602::4522205::INSTR', visa_library = 'C:/windows/System32/visa64.dll', timeout = 100000)
keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_OFF
keithley_instrument.smub.source.output = keithley_instrument.smub.OUTPUT_OFF

In [81]:
keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_OFF

In [86]:
#-- Reset SourceMeter instrument to default conditions.
keithley_instrument.reset()

In [87]:
#-- Clear the front panel display then prompt for input parameters if missing. 
keithley_instrument.display.clear() 

In [ ]:
# -- Generate a single pulse with the following characteristics:
# -- * Bias (idle) level = 0 V
# -- * Pulse level = 0.2 V
# -- * Pulse width = 20 ms
# -- Configure the source function.
bias = 0
pulse_voltage = 0.2
ton = 0.2 # [s], pulse on time
toff = 0.8 # [s], pulse off time
n_pulses = 20 
# -- Update display with test info. 
keithley_instrument.display.settext("PulseV")  #-- Line 1 (20 characters max) 

#-- Configure source and measure settings (drain). 

keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_OFF 

if abs(pulse_voltage) > abs(bias):

    keithley_instrument.smua.source.rangev = pulse_voltage 

else: 

    keithley_instrument.smua.source.rangev = bias 

keithley_instrument.smua.source.func = keithley_instrument.smua.OUTPUT_DCVOLTS
#-- Set the voltage source range and the idle or bias source level and limit.
keithley_instrument.smua.source.levelv = pulse_voltage
keithley_instrument.smua.source.limiti = 0.1

In [89]:
# -- Use trigger timer 1 to control the period and trigger timer 2 to control the 
# -- pulse width. Alias the timers for convenience and clarity.
# period_timer = keithley_instrument.trigger.timer[1]
# pulsewidth_timer = keithley_instrument.trigger.timer[2]
# -- Configure the period timer to output 10 total trigger events.
keithley_instrument.trigger.timer[1].delay = 1
# -- The effective count is 10 because the passthrough setting is true.
keithley_instrument.trigger.timer[1].count = 9
# -- Configure the timer to immediately output a trigger event when it is started.
keithley_instrument.trigger.timer[1].passthrough = True
# -- Start the timer when the SMU moves from the ARM layer to the TRIGGER layer.
keithley_instrument.trigger.timer[1].stimulus = keithley_instrument.smua.trigger.ARMED_EVENT_ID
# -- Configure the pulse width timer to output one trigger event for each period.
keithley_instrument.trigger.timer[2].delay = 0.5
keithley_instrument.trigger.timer[2].count = 1
# -- Do not immediately output a trigger event when pulse width timer is started.
keithley_instrument.trigger.timer[2].passthrough = False
# -- Start the pulse width timer with the period timer output trigger event.
keithley_instrument.trigger.timer[2].stimulus = keithley_instrument.trigger.timer[1].EVENT_ID
# -- Configure the trigger model to execute a 10-point fixed-level voltage pulse 
# -- train. No measurements are made.
keithley_instrument.smua.trigger.source.listv({0.2})
keithley_instrument.smua.trigger.source.action = keithley_instrument.smua.ENABLE
keithley_instrument.smua.trigger.measure.action = keithley_instrument.smua.DISABLE
# -- Set the trigger source limit, which can be different than the bias limit.
# -- This is an important setting for pulsing in the extended operating area.
keithley_instrument.smua.trigger.source.limiti = 1
keithley_instrument.smua.measure.rangei = 1
# -- Trigger SMU source action with the period timer event.
keithley_instrument.smua.trigger.source.stimulus = keithley_instrument.trigger.timer[1].EVENT_ID
# -- Configure the endpulse action to achieve a pulse.
keithley_instrument.smua.trigger.endpulse.action = keithley_instrument.smua.SOURCE_IDLE
# -- Trigger the SMU end pulse action with a pulse width timer event.
keithley_instrument.smua.trigger.endpulse.stimulus = keithley_instrument.trigger.timer[2].EVENT_ID
# -- Set the trigger model count to generate one 10-point pulse train.
keithley_instrument.smua.trigger.arm.count = 1
keithley_instrument.smua.trigger.count = 10
# -- Turn on the SMU output and initiate the trigger model to output the pulse train.
keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_ON
keithley_instrument.smua.trigger.initiate()
# # -- Wait for the sweep to complete.
keithley_instrument.waitcomplete()
# -- Turn off SMU output.
keithley_instrument.smua.source.output = keithley_instrument.smua.OUTPUT_OFF

# Pyvisa

In [9]:
import pyvisa

In [10]:
rm = pyvisa.ResourceManager('C:/windows/System32/visa64.dll')
print(rm.list_resources())


('USB0::0x05E6::0x2602::4522205::INSTR', 'ASRL3::INSTR')


In [11]:
smu = rm.open_resource('USB0::0x05E6::0x2602::4522205::INSTR')

In [12]:
print(smu.query("*IDN?"))

Keithley Instruments Inc., Model 2602B, 4522205, 3.3.5



In [73]:
#-- Reset SourceMeter instrument to default conditions.
smu.write('reset()')

9

In [74]:
#-- Clear the front panel display then prompt for input parameters if missing. 
smu.write('display.clear()')

17

In [66]:
# smu.write('smua.source.func = smua.OUTPUT_DCVOLTS')

In [67]:
# smu.write('smua.source.output = smua.OUTPUT_ON')

In [68]:
# smu.write('smua.source.output = smua.OUTPUT_OFF')

In [69]:
# value = 0.2
# smu.write(f"smua.source.levelv = {value}")

In [75]:
# -- Generate a single pulse with the following characteristics:
# -- * Bias (idle) level = 0 V
# -- * Pulse level = 0.2 V
# -- * Pulse width = 20 ms
# -- Configure the source function.
bias = 0
pulse_voltage = 0.2
ton = 0.2 # [s], pulse on time
toff = 0.8 # [s], pulse off time
n_pulses = 20 
# -- Update display with test info. 
smu.write('display.settext("PulseV")')


#-- Configure source and measure settings (drain). 
smu.write('smua.source.output = smua.OUTPUT_OFF')

if abs(pulse_voltage) > abs(bias):
    smu.write(f"smua.source.rangev = {pulse_voltage}") 
else: 
    smu.write(f"smua.source.rangev = {bias}")

smu.write(f"smua.source.func = smua.OUTPUT_DCVOLTS")
#-- Set the voltage source range and the idle or bias source level and limit.
smu.write(f"smua.source.levelv = {pulse_voltage}")
smu.write(f"smua.source.limiti = {1}")

24

In [76]:
# -- Use trigger timer 1 to control the period and trigger timer 2 to control the 
# -- pulse width. Alias the timers for convenience and clarity.
# period_timer = keithley_instrument.trigger.timer[1]
# pulsewidth_timer = keithley_instrument.trigger.timer[2]
# -- Configure the period timer to output 10 total trigger events.
smu.write(f"trigger.timer[1].delay = 1")
# -- The effective count is 10 because the passthrough setting is true.
smu.write(f"trigger.timer[1].count = 9")
# -- Configure the timer to immediately output a trigger event when it is started.
smu.write(f"trigger.timer[1].passthrough = true")
# -- Start the timer when the SMU moves from the ARM layer to the TRIGGER layer.
smu.write(f"trigger.timer[1].stimulus = smua.trigger.ARMED_EVENT_ID")
# -- Configure the pulse width timer to output one trigger event for each period.
smu.write(f"trigger.timer[2].delay = 0.5")
smu.write(f"trigger.timer[2].count = 1")
# -- Do not immediately output a trigger event when pulse width timer is started.
smu.write(f"trigger.timer[2].passthrough = false")
# -- Start the pulse width timer with the period timer output trigger event.
smu.write(f"trigger.timer[2].stimulus = trigger.timer[1].EVENT_ID")


55

In [77]:
# -- Configure the trigger model to execute a 10-point fixed-level voltage pulse 
# -- train. No measurements are made.
smu.write("smua.trigger.source.listv({0.2})")
smu.write(f"smua.trigger.source.action = smua.ENABLE")
smu.write(f"smua.trigger.measure.action = smua.DISABLE")
# -- Set the trigger source limit, which can be different than the bias limit.
# -- This is an important setting for pulsing in the extended operating area.
smu.write(f"smua.trigger.source.limiti = 1")
smu.write(f"smua.measure.rangei = 1")
# -- Trigger SMU source action with the period timer event.
smu.write(f"smua.trigger.source.stimulus = trigger.timer[1].EVENT_ID")
# -- Configure the endpulse action to achieve a pulse.
smu.write(f"smua.trigger.endpulse.action = smua.SOURCE_IDLE")
# -- Trigger the SMU end pulse action with a pulse width timer event.
smu.write(f"smua.trigger.endpulse.stimulus = trigger.timer[2].EVENT_ID")
# -- Set the trigger model count to generate one 10-point pulse train.
smu.write(f"smua.trigger.arm.count = 1")
smu.write(f"smua.trigger.count = 10")

25

In [78]:

# -- Turn on the SMU output and initiate the trigger model to output the pulse train.
smu.write(f"smua.source.output = smua.OUTPUT_ON")
smu.write(f"smua.trigger.initiate()")
# # -- Wait for the sweep to complete.
smu.write(f"waitcomplete()")


16

In [79]:
# -- Turn off SMU output.
smu.write(f"mua.source.output = smua.OUTPUT_OFF")

37